# Transportation Research Part C (v3) — Lightweight Correlation-Weighted T-GCN & Unified HCM 2010 Delay Proof

### Why v3?
In v2, empirical ablations showed two key findings for short-horizon ($T=6$), small-$N$ ($N=4$) heterogeneous intersection graphs:
1. **Recurrent temporal modeling (GRU) outperforms unpadded 1D-CNN ST-GCN** because unpadded convolutions truncate $T=6 \to 4 \to 2$, discarding early sequence history.
2. **Overparameterized learnable $E_1 E_2^T$ node embeddings overfit** on 263 training windows, whereas the **empirical training-set correlation graph ($\hat{A}_{\text{corr}}$)** generalizes cleanly.

### Proposed Architecture (v3): **Lightweight Correlation-Weighted T-GCN**
- **Gated Correlation-GCN:** Single learnable scalar gate $\alpha = \sigma(\gamma)$ balancing self-node preservation ($I$) and empirical correlation propagation ($\hat{A}_{\text{corr}}$) — zero $E_1 E_2^T$ overfitting, zero complete-graph oversmoothing.
- **Full-Sequence GRU Decoder + Cyclic Temporal Encoding:** Preserves all 6 lookback steps with $\sin/\cos$ time-of-day and day-of-week features (and no noisy Persistence skip connection).
- **Joint Multimodal + PCU Capacity Loss** (`Huber(veh) + 0.35 * Huber(pcu)`).
- **3-Seed Statistical Rigor (`42, 123, 2026`):** Reports `Mean ± Std` across 3 seeds for all neural models, plus the unified **HCM 2010 Oversaturated 2-Phase Control Delay Proof** at the end.

In [ ]:
%matplotlib inline
import os
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. DEVICE & PATHS ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"GPU Acceleration Active: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU.")

DEFAULT_REPO_DATA = os.path.abspath(os.path.join("..", "data", "csv_5min"))
LOCAL_FALLBACK = r"D:\RF-DETR (Thesis)\Extracted Frames\CSV1"
CSV_DIRECTORY = DEFAULT_REPO_DATA if (os.path.isdir(DEFAULT_REPO_DATA) and glob.glob(os.path.join(DEFAULT_REPO_DATA, "*_flow_data.csv"))) else LOCAL_FALLBACK
OUT_DIR = os.path.abspath(os.path.join("..", "results", "reproduced_v3")) if CSV_DIRECTORY == DEFAULT_REPO_DATA else os.path.join(CSV_DIRECTORY, "TRC_Authentic_Results_v3")
os.makedirs(OUT_DIR, exist_ok=True)

# Hyperparameters
SEEDS = [42, 123, 2026]
LOOKBACK = 6        # T = 6 steps (30 mins at 5-min bins)
HORIZON = 3         # Q = 3 steps (5-min, 10-min, 15-min ahead)
BATCH_SIZE = 16
EPOCHS = 120
EARLY_STOP_PATIENCE = 20
LR = 1.2e-3
LAMBDA_PCU = 0.35   # Balanced joint PCU capacity loss weight

print(f"Input CSV Directory: {CSV_DIRECTORY}")
print(f"Output Directory:    {OUT_DIR}")

## 2. Leak-Free Chronological Dataset, Per-Node Normalization & Empirical Correlation Graph

In [ ]:
NODE_MAPPING = {"C&B": 0, "FireService": 1, "Circuit": 2, "Lokkhipur": 3}
NODE_NAMES = ["C&B (South)", "Fire Service (North)", "Circuit House (East)", "Lokkhipur (West)"]

ACTIVE_CLASSES = [
    "bicycle", "bike", "bus", "car", "cng", "easybike",
    "leguna", "rickshaw", "tractor", "truck", "van", "wheelbarrow"
]
NUM_NODES = 4
NUM_CLASSES = len(ACTIVE_CLASSES)       # 12 active land-based modes
IN_CHANNELS = NUM_CLASSES * 2           # 24 channels (12 Inflow + 12 Outflow)
TIME_DIM = 4                            # sin/cos(time_of_day), sin/cos(day_of_week)

PCU_FACTORS = {
    "easybike": 2.0, "cng": 2.0, "rickshaw": 2.0, "bike": 0.75,
    "bicycle": 0.5, "car": 1.0, "bus": 3.0, "van": 2.0,
    "truck": 3.0, "leguna": 1.0, "tractor": 2.0, "wheelbarrow": 1.5
}

PCU_VEC = np.zeros((1, 1, 1, IN_CHANNELS), dtype=np.float32)
for i, cls in enumerate(ACTIVE_CLASSES):
    PCU_VEC[0, 0, 0, i] = PCU_FACTORS[cls]
    PCU_VEC[0, 0, 0, i + NUM_CLASSES] = PCU_FACTORS[cls]

def encode_timestamp(ts):
    ts = pd.Timestamp(ts)
    mins = ts.hour * 60.0 + ts.minute + ts.second / 60.0
    tod = 2.0 * np.pi * (mins / 1440.0)
    dow = 2.0 * np.pi * (ts.dayofweek / 7.0)
    return np.array([np.sin(tod), np.cos(tod), np.sin(dow), np.cos(dow)], dtype=np.float32)

def load_chronological_sequences(csv_dir, lookback=6, horizon=3, max_gap_minutes=15):
    all_files = sorted(glob.glob(os.path.join(csv_dir, "*_flow_data.csv")))
    rfdetr_files = [
        f for f in all_files
        if "_yolo_" not in os.path.basename(f).lower()
        and "_backup_" not in os.path.basename(f).lower()
    ]
    if not rfdetr_files:
        raise FileNotFoundError(f"No RF-DETR *_flow_data.csv files found in {csv_dir}")

    dfs = [pd.read_csv(f) for f in rfdetr_files]
    full_df = pd.concat(dfs, ignore_index=True)
    full_df["timestamp"] = pd.to_datetime(full_df["timestamp"])
    full_df = full_df.sort_values("timestamp").reset_index(drop=True)
    full_df = full_df.drop_duplicates(subset=["timestamp", "node_id", "flow_direction"], keep="last")

    unique_ts = sorted(full_df["timestamp"].unique())
    print(f"Loaded {len(rfdetr_files)} CSV files | {len(unique_ts)} unique 5-min intervals.")

    sessions = []
    current_session = [unique_ts[0]]
    for ts in unique_ts[1:]:
        if (ts - current_session[-1]).total_seconds() / 60.0 <= max_gap_minutes:
            current_session.append(ts)
        else:
            sessions.append(current_session)
            current_session = [ts]
    sessions.append(current_session)
    print(f"Stitched into {len(sessions)} continuous recording sessions.")

    seq_len = lookback + horizon
    X_list, Y_list, T_list, ts_list = [], [], [], []

    for sess in sessions:
        if len(sess) < seq_len:
            continue
        sess_tensor = np.zeros((len(sess), NUM_NODES, IN_CHANNELS), dtype=np.float32)
        sess_time   = np.zeros((len(sess), TIME_DIM), dtype=np.float32)
        for t_idx, ts in enumerate(sess):
            sess_time[t_idx] = encode_timestamp(ts)
            sub = full_df[full_df["timestamp"] == ts]
            for _, row in sub.iterrows():
                nid = row["node_id"]
                if nid not in NODE_MAPPING:
                    continue
                n_idx = NODE_MAPPING[nid]
                vals = row[ACTIVE_CLASSES].values.astype(np.float32)
                if row["flow_direction"] == "Inflow":
                    sess_tensor[t_idx, n_idx, :NUM_CLASSES] = vals
                elif row["flow_direction"] == "Outflow":
                    sess_tensor[t_idx, n_idx, NUM_CLASSES:] = vals

        for start in range(len(sess) - seq_len + 1):
            X_list.append(sess_tensor[start : start + lookback])
            Y_list.append(sess_tensor[start + lookback : start + seq_len])
            T_list.append(sess_time[start : start + lookback])
            ts_list.append(sess[start + lookback])

    return (
        np.array(X_list, dtype=np.float32),
        np.array(Y_list, dtype=np.float32),
        np.array(T_list, dtype=np.float32),
        ts_list
    )

class TrafficDatasetV3(Dataset):
    def __init__(self, X, Y, T_feat, mean, std):
        self.X = (X - mean) / std
        self.Y = (Y - mean) / std
        self.T = T_feat

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.float32),
            torch.tensor(self.T[idx], dtype=torch.float32),
            torch.tensor(self.Y[idx], dtype=torch.float32)
        )

X_all, Y_all, T_all, ts_all = load_chronological_sequences(CSV_DIRECTORY, LOOKBACK, HORIZON)
n_total = len(X_all)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)

X_train, Y_train, T_train = X_all[:n_train], Y_all[:n_train], T_all[:n_train]
X_val,   Y_val,   T_val   = X_all[n_train:n_train+n_val], Y_all[n_train:n_train+n_val], T_all[n_train:n_train+n_val]
X_test,  Y_test,  T_test  = X_all[n_train+n_val:], Y_all[n_train+n_val:], T_all[n_train+n_val:]
ts_test = ts_all[n_train+n_val:]

print(f"Chronological Split -> Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Test Period: {ts_test[0]} to {ts_test[-1]}")

# Per-node Z-score normalization fit ONLY on Train
mean = X_train.mean(axis=(0, 1), keepdims=True)
std  = X_train.std(axis=(0, 1), keepdims=True)
std[std < 1e-4] = 1.0

mean_t    = torch.tensor(mean, dtype=torch.float32, device=device)
std_t     = torch.tensor(std,  dtype=torch.float32, device=device)
pcu_vec_t = torch.tensor(PCU_VEC[..., :NUM_CLASSES], dtype=torch.float32, device=device)

train_pcu = (Y_train[..., :NUM_CLASSES] * PCU_VEC[..., :NUM_CLASSES]).sum(axis=-1)
pcu_scale_t = torch.tensor(float(max(train_pcu.std(), 1.0)), dtype=torch.float32, device=device)

# Adjacency matrices
A_UNIFORM  = torch.full((NUM_NODES, NUM_NODES), 0.25, dtype=torch.float32, device=device)
A_IDENTITY = torch.eye(NUM_NODES, dtype=torch.float32, device=device)

train_in_pcu_flat = (X_train[:, -1, :, :NUM_CLASSES] * PCU_VEC[0, 0, :, :NUM_CLASSES]).sum(axis=-1)
corr_mat = np.nan_to_num(np.corrcoef(train_in_pcu_flat.T), nan=0.0)
corr_pos = np.maximum(corr_mat, 0.0)
np.fill_diagonal(corr_pos, 1.0)
corr_row_norm = corr_pos / np.maximum(corr_pos.sum(axis=1, keepdims=True), 1e-6)
A_CORR = torch.tensor(corr_row_norm, dtype=torch.float32, device=device)

print("\nEmpirical Training-Set Correlation Adjacency (A_CORR):")
print(pd.DataFrame(corr_row_norm.round(3), index=NODE_NAMES, columns=NODE_NAMES))

## 3. Model Definitions (Baselines, Ablations & Proposed Lightweight Correlation-Weighted T-GCN v3)

In [ ]:
class TemporalConvLayer(nn.Module):
    def __init__(self, c_in, c_out, kernel_size=3):
        super().__init__()
        self.c_out = c_out
        self.conv = nn.Conv2d(c_in, 2 * c_out, kernel_size=(kernel_size, 1))

    def forward(self, x):
        h = self.conv(x)
        p, q = torch.split(h, self.c_out, dim=1)
        return p * torch.sigmoid(q)

class StaticSpatialConvLayer(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(c_in, c_out))
        self.bias   = nn.Parameter(torch.zeros(c_out))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj):
        x_perm = x.permute(0, 2, 3, 1)
        agg = torch.einsum("ij,btjc->btic", adj, x_perm)
        out = torch.matmul(agg, self.weight) + self.bias
        return out.permute(0, 3, 1, 2)

class StaticSTGCN(nn.Module):
    def __init__(self, num_nodes, in_channels, lookback, horizon, adj):
        super().__init__()
        self.register_buffer("adj", adj)
        self.tcn1 = TemporalConvLayer(in_channels, 32)
        self.gcn  = StaticSpatialConvLayer(32, 32)
        self.tcn2 = TemporalConvLayer(32, 64)
        self.ln   = nn.LayerNorm(64)
        self.drop = nn.Dropout(0.15)
        self.res  = nn.Conv2d(in_channels, 64, kernel_size=(1, 1))
        self.fc_time = nn.Linear(lookback - 4, horizon)
        self.fc_ch   = nn.Linear(64, in_channels)

    def forward(self, x, t_feat=None):
        x_p = x.permute(0, 3, 1, 2)
        t1 = self.tcn1(x_p)
        s  = self.drop(F.relu(self.gcn(t1, self.adj)))
        t2 = self.tcn2(s)
        r  = self.res(x_p[:, :, -t2.shape[2]:, :])
        h  = self.ln((t2 + r).permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        h  = self.fc_time(h.permute(0, 1, 3, 2))
        return self.fc_ch(h.permute(0, 3, 2, 1))

class LSTMModel(nn.Module):
    def __init__(self, num_nodes, in_channels, horizon, hidden_dim=128):
        super().__init__()
        self.horizon = horizon
        self.in_channels = in_channels
        self.lstm = nn.LSTM(num_nodes * in_channels, hidden_dim, num_layers=2, batch_first=True, dropout=0.15)
        self.fc   = nn.Linear(hidden_dim, horizon * num_nodes * in_channels)

    def forward(self, x, t_feat=None):
        B, T, N, C = x.shape
        out, _ = self.lstm(x.reshape(B, T, N * C))
        return self.fc(out[:, -1, :]).reshape(B, self.horizon, N, self.in_channels)

class StandardTGCN(nn.Module):
    def __init__(self, num_nodes, in_channels, horizon, adj, hidden_dim=64):
        super().__init__()
        self.register_buffer("adj", adj)
        self.horizon = horizon
        self.in_channels = in_channels
        self.gcn = StaticSpatialConvLayer(in_channels, hidden_dim)
        self.gru = nn.GRU(num_nodes * hidden_dim, hidden_dim * 2, batch_first=True)
        self.fc  = nn.Linear(hidden_dim * 2, horizon * num_nodes * in_channels)

    def forward(self, x, t_feat=None):
        B, T, N, C = x.shape
        g_out = F.relu(self.gcn(x.permute(0, 3, 1, 2), self.adj)).permute(0, 2, 3, 1)
        gru_out, _ = self.gru(g_out.reshape(B, T, -1))
        return self.fc(gru_out[:, -1, :]).reshape(B, self.horizon, N, self.in_channels)

class GatedCorrelationGCN(nn.Module):
    """Single-gate spatial convolution: alpha * W_self(X) + (1 - alpha) * W_corr(A_corr @ X)."""
    def __init__(self, c_in, c_out, adj_corr):
        super().__init__()
        self.register_buffer("adj_corr", adj_corr)
        self.w_self = nn.Linear(c_in, c_out, bias=False)
        self.w_corr = nn.Linear(c_in, c_out, bias=False)
        self.bias   = nn.Parameter(torch.zeros(c_out))
        self.alpha_logit = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        alpha = torch.sigmoid(self.alpha_logit)
        h_self = self.w_self(x)
        h_corr = self.w_corr(torch.einsum("ij,btjc->btic", self.adj_corr, x))
        return F.relu(alpha * h_self + (1.0 - alpha) * h_corr + self.bias)

class CorrelationTGCN_v3(nn.Module):
    """
    Proposed Lightweight Correlation-Weighted T-GCN (v3):
    Gated Correlation-GCN + Full-Sequence GRU + Cyclic Time-of-Day/Day-of-Week Encoding.
    """
    def __init__(self, num_nodes, in_channels, time_dim, horizon, adj_corr, gcn_dim=48, gru_dim=128):
        super().__init__()
        self.num_nodes = num_nodes
        self.in_channels = in_channels
        self.horizon = horizon

        self.gcn  = GatedCorrelationGCN(in_channels + time_dim, gcn_dim, adj_corr)
        self.ln   = nn.LayerNorm(gcn_dim)
        self.drop = nn.Dropout(0.20)
        self.gru  = nn.GRU(num_nodes * gcn_dim, gru_dim, num_layers=1, batch_first=True)
        self.fc   = nn.Linear(gru_dim, horizon * num_nodes * in_channels)

    def forward(self, x, t_feat):
        B, T, N, C = x.shape
        t_exp = t_feat.unsqueeze(2).expand(B, T, N, -1)
        x_in  = torch.cat([x, t_exp], dim=-1)
        g_out = self.drop(self.ln(self.gcn(x_in)))
        gru_out, _ = self.gru(g_out.reshape(B, T, -1))
        out = self.fc(self.drop(gru_out[:, -1, :]))
        return out.reshape(B, self.horizon, N, self.in_channels)

## 4. 3-Seed Multi-Run Benchmark & Ablation Study (Mean ± Std)
Runs every neural model across 3 independent random seeds (`42, 123, 2026`) with early stopping (`patience=20`) so all reported metrics are statistically stable and reviewer-proof.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def joint_loss(pred_norm, target_norm, lambda_pcu=0.0):
    veh_loss = F.huber_loss(pred_norm, target_norm, delta=1.0)
    if lambda_pcu <= 0.0:
        return veh_loss
    p_den = pred_norm * std_t + mean_t
    y_den = target_norm * std_t + mean_t
    p_pcu = (p_den[..., :NUM_CLASSES] * pcu_vec_t).sum(dim=-1) / pcu_scale_t
    y_pcu = (y_den[..., :NUM_CLASSES] * pcu_vec_t).sum(dim=-1) / pcu_scale_t
    return veh_loss + lambda_pcu * F.huber_loss(p_pcu, y_pcu, delta=1.0)

def compute_all_metrics(y_true, y_pred):
    veh_mae  = np.mean(np.abs(y_true - y_pred))
    veh_rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    true_in_pcu = (y_true[..., :NUM_CLASSES] * PCU_VEC[..., :NUM_CLASSES]).sum(axis=-1)
    pred_in_pcu = (y_pred[..., :NUM_CLASSES] * PCU_VEC[..., :NUM_CLASSES]).sum(axis=-1)
    pcu_mae  = np.mean(np.abs(true_in_pcu - pred_in_pcu))
    pcu_rmse = np.sqrt(np.mean((true_in_pcu - pred_in_pcu) ** 2))
    mask = true_in_pcu >= 5.0
    pcu_mape = np.mean(np.abs((true_in_pcu[mask] - pred_in_pcu[mask]) / true_in_pcu[mask])) * 100.0
    h_mae = [np.mean(np.abs(true_in_pcu[:, h, :] - pred_in_pcu[:, h, :])) for h in range(y_true.shape[1])]
    return {
        "Veh_MAE": veh_mae, "Veh_RMSE": veh_rmse,
        "PCU_MAE": pcu_mae, "PCU_RMSE": pcu_rmse, "PCU_MAPE(%)": pcu_mape,
        "H1_5min_PCU_MAE": h_mae[0], "H2_10min_PCU_MAE": h_mae[1], "H3_15min_PCU_MAE": h_mae[2]
    }

def run_single_seed(model_fn, seed, lambda_pcu=0.0):
    set_seed(seed)
    tr_loader = DataLoader(TrafficDatasetV3(X_train, Y_train, T_train, mean, std), batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(TrafficDatasetV3(X_val,   Y_val,   T_val,   mean, std), batch_size=BATCH_SIZE, shuffle=False)
    te_loader = DataLoader(TrafficDatasetV3(X_test,  Y_test,  T_test,  mean, std), batch_size=BATCH_SIZE, shuffle=False)

    model = model_fn().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=3e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=8)

    best_val = float("inf")
    best_state = None
    no_improve = 0

    for _ in range(EPOCHS):
        model.train()
        for xb, tb, yb in tr_loader:
            xb, tb, yb = xb.to(device), tb.to(device), yb.to(device)
            opt.zero_grad()
            loss = joint_loss(model(xb, tb), yb, lambda_pcu=lambda_pcu)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for xv, tv, yv in va_loader:
                xv, tv, yv = xv.to(device), tv.to(device), yv.to(device)
                v_loss += joint_loss(model(xv, tv), yv, lambda_pcu=lambda_pcu).item() * xv.size(0)
        v_loss /= len(va_loader.dataset)
        sched.step(v_loss)

        if v_loss < best_val - 1e-5:
            best_val = v_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    preds_all = []
    with torch.no_grad():
        for xt, tt, _ in te_loader:
            xt, tt = xt.to(device), tt.to(device)
            p_den = torch.clamp(model(xt, tt) * std_t + mean_t, min=0.0)
            preds_all.append(p_den.cpu().numpy())
    return np.concatenate(preds_all, axis=0), best_state, model

# Deterministic baselines
mean_results = {}
formatted_results = {}
saved_preds = {}

ha_pred = np.repeat(Y_train.mean(axis=0, keepdims=True), len(Y_test), axis=0)
pers_pred = np.repeat(X_test[:, -1:, :, :], HORIZON, axis=1)

for name, pred in [("Historical Average (HA)", ha_pred), ("Persistence (Last-Step)", pers_pred)]:
    m = compute_all_metrics(Y_test, pred)
    mean_results[name] = m
    formatted_results[name] = {k: f"{v:.4f} ± 0.0000" for k, v in m.items()}

neural_specs = [
    ("LSTM",                             lambda: LSTMModel(NUM_NODES, IN_CHANNELS, HORIZON), 0.0),
    ("Static ST-GCN (Uniform 0.25)",     lambda: StaticSTGCN(NUM_NODES, IN_CHANNELS, LOOKBACK, HORIZON, A_UNIFORM), 0.0),
    ("ST-GCN (w/o Graph, A=I)",          lambda: StaticSTGCN(NUM_NODES, IN_CHANNELS, LOOKBACK, HORIZON, A_IDENTITY), 0.0),
    ("Standard T-GCN (A_corr)",          lambda: StandardTGCN(NUM_NODES, IN_CHANNELS, HORIZON, A_CORR), 0.0),
    ("Proposed Corr-TGCN (v3)",          lambda: CorrelationTGCN_v3(NUM_NODES, IN_CHANNELS, TIME_DIM, HORIZON, A_CORR), LAMBDA_PCU),
]

best_v3_alpha = None
for idx_m, (name, fn, l_pcu) in enumerate(neural_specs, start=3):
    print(f"{idx_m}/7 Running 3-seed evaluation for {name}...")
    seed_metrics = []
    seed_preds = []
    for s in SEEDS:
        p, st, mdl = run_single_seed(fn, s, lambda_pcu=l_pcu)
        seed_metrics.append(compute_all_metrics(Y_test, p))
        seed_preds.append(p)
        if name == "Proposed Corr-TGCN (v3)" and s == SEEDS[0]:
            torch.save(st, os.path.join(OUT_DIR, "best_corr_tgcn_v3.pth"))
            best_v3_alpha = float(torch.sigmoid(mdl.gcn.alpha_logit).item())

    df_s = pd.DataFrame(seed_metrics)
    mean_results[name] = df_s.mean().to_dict()
    formatted_results[name] = {
        col: f"{df_s[col].mean():.4f} ± {df_s[col].std():.4f}" for col in df_s.columns
    }
    saved_preds[name] = np.mean(seed_preds, axis=0)  # 3-seed mean prediction for smooth plotting & control

summary_mean_df = pd.DataFrame(mean_results).T.round(4)
summary_fmt_df  = pd.DataFrame(formatted_results).T

print("\n" + "=" * 106)
print("v3 AUTHENTIC 3-SEED CHRONOLOGICAL BENCHMARK & ABLATION TABLE (MEAN ACROSS SEEDS 42, 123, 2026)")
print("=" * 106)
print(summary_mean_df.to_string())
print("=" * 106)
print(f"Learned Self-Gate Weight (alpha) in Proposed Corr-TGCN (v3): {best_v3_alpha:.3f} (Self) vs {1.0 - best_v3_alpha:.3f} (A_corr)")

# Save benchmark tables & predictions
summary_mean_df.to_csv(os.path.join(OUT_DIR, "TRC_v3_Benchmark_Table.csv"))
summary_fmt_df.to_csv(os.path.join(OUT_DIR, "TRC_v3_Benchmark_MeanStd.csv"))

v3_pred = saved_preds["Proposed Corr-TGCN (v3)"]
np.savez(
    os.path.join(OUT_DIR, "authentic_test_predictions_v3.npz"),
    y_true=Y_test,
    corr_tgcn_v3_pred=v3_pred,
    std_tgcn_pred=saved_preds["Standard T-GCN (A_corr)"],
    lstm_pred=saved_preds["LSTM"],
    static_stgcn_pred=saved_preds["Static ST-GCN (Uniform 0.25)"]
)

## 5. Unified HCM 2010 Oversaturated 2-Phase Control Delay Proof (Using v3 Predictions)
Evaluates Fixed 50/50, Fixed 65/35, Fixed 75/25, **Proposed Correlation-Weighted T-GCN (v3)**, and the **Oracle Perfect-Foresight Bound** on the 58 unseen test windows.

In [ ]:
pcu_w = np.array([PCU_FACTORS[c] for c in ACTIVE_CLASSES], dtype=np.float32)
q_true_hr = (Y_test[..., :12]  * pcu_w).sum(axis=-1).sum(axis=1) * 4.0
q_pred_hr = (v3_pred[..., :12] * pcu_w).sum(axis=-1).sum(axis=1) * 4.0

S_SAT = np.array([3000.0, 3000.0, 1500.0, 3000.0])
L_LOST = 8.0

def hcm2010_los(d):
    if d <= 10.0: return "A"
    if d <= 20.0: return "B"
    if d <= 35.0: return "C"
    if d <= 55.0: return "D"
    if d <= 80.0: return "E"
    return "F"

def hcm2010_delay(q_actual_hr, C_cycle, g_per_leg, s_sat, T=0.25, k=0.50, I=1.0):
    lam = g_per_leg / C_cycle
    cap = np.maximum(s_sat * lam, 1.0)
    X   = q_actual_hr / cap
    X_u = np.minimum(X, 1.0)
    d1 = (0.5 * C_cycle * (1.0 - lam) ** 2) / np.maximum(1.0 - lam * X_u, 0.05)
    d2 = 900.0 * T * ((X - 1.0) + np.sqrt((X - 1.0) ** 2 + (8.0 * k * I * X) / (cap * T)))
    d_i = d1 + d2
    return np.sum(d_i * q_actual_hr) / np.maximum(np.sum(q_actual_hr), 1.0)

def compute_2phase_webster(q_plan_hr, s_sat, L_lost=8.0, C_min=60.0, C_max=150.0, g_min=15.0):
    y_ns = max(q_plan_hr[0] / s_sat[0], q_plan_hr[1] / s_sat[1])
    y_ew = max(q_plan_hr[2] / s_sat[2], q_plan_hr[3] / s_sat[3])
    Y = np.clip(y_ns + y_ew, 0.15, 0.90)
    C_opt = np.clip((1.5 * L_lost + 5.0) / (1.0 - Y), C_min, C_max)
    G_eff = C_opt - L_lost
    g_ns = np.clip((y_ns / max(y_ns + y_ew, 1e-6)) * G_eff, g_min, G_eff - g_min)
    g_ew = G_eff - g_ns
    return C_opt, g_ns, g_ew, np.array([g_ns, g_ns, g_ew, g_ew])

delay_rows = []
for idx in range(len(q_true_hr)):
    q_t, q_p = q_true_hr[idx], q_pred_hr[idx]
    if q_t.sum() < 100.0:
        continue
    d50 = hcm2010_delay(q_t, 120.0, np.array([56.0, 56.0, 56.0, 56.0]), S_SAT)
    d65 = hcm2010_delay(q_t, 120.0, np.array([72.8, 72.8, 39.2, 39.2]), S_SAT)
    d75 = hcm2010_delay(q_t, 120.0, np.array([84.0, 84.0, 28.0, 28.0]), S_SAT)
    C_a, g_ns, g_ew, g_a = compute_2phase_webster(q_p, S_SAT, L_LOST)
    d_a = hcm2010_delay(q_t, C_a, g_a, S_SAT)
    C_o, _, _, g_o = compute_2phase_webster(q_t, S_SAT, L_LOST)
    d_o = hcm2010_delay(q_t, C_o, g_o, S_SAT)
    delay_rows.append({
        "Window": idx, "Fixed_50_50": d50, "Fixed_65_35": d65, "Fixed_75_25": d75,
        "Corr_TGCN_v3": d_a, "Oracle_Bound": d_o,
        "Cycle_s": C_a, "g_NS_s": g_ns, "g_EW_s": g_ew
    })

delay_df = pd.DataFrame(delay_rows)
delay_df.to_csv(os.path.join(OUT_DIR, "TRC_v3_HCM2010_Delay_Evaluation.csv"), index=False)

mean_v3 = delay_df["Corr_TGCN_v3"].mean()
strat_rows = []
for label, col in [
    ("Fixed-Time 50/50 (C=120s)",        "Fixed_50_50"),
    ("Fixed-Time 65/35 (C=120s)",        "Fixed_65_35"),
    ("Fixed-Time 75/25 (C=120s)",        "Fixed_75_25"),
    ("Proposed Corr-TGCN (v3) Adaptive", "Corr_TGCN_v3"),
    ("Oracle Perfect-Foresight Bound",   "Oracle_Bound")
]:
    md = delay_df[col].mean()
    sav = ((md - mean_v3) / md) * 100.0 if col != "Corr_TGCN_v3" else 0.0
    strat_rows.append({
        "Strategy": label,
        "Mean_Delay(s/veh)": round(md, 2),
        "Median_Delay(s/veh)": round(delay_df[col].median(), 2),
        "P95_Delay(s/veh)": round(np.percentile(delay_df[col], 95), 2),
        "HCM_LOS": hcm2010_los(md),
        "v3_Savings(%)": round(sav, 2)
    })

strat_df = pd.DataFrame(strat_rows)
strat_df.to_csv(os.path.join(OUT_DIR, "TRC_v3_Signal_Strategy_Summary.csv"), index=False)

print("\n" + "=" * 95)
print("v3 HCM 2010 2-PHASE ARTERIAL CONTROL DELAY COMPARISON (58 UNSEEN TEST WINDOWS)")
print("=" * 95)
print(strat_df.to_string(index=False))
print("=" * 95)

## 6. Publication-Grade v3 Figures (300 DPI, White Background)

In [ ]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#F8F9FA",
    "axes.edgecolor": "#CCCCCC",
    "axes.grid": True,
    "grid.color": "#E5E7EB",
    "font.size": 11
})

true_in_pcu = (Y_test[..., :NUM_CLASSES] * PCU_VEC[..., :NUM_CLASSES]).sum(axis=-1)
v3_in_pcu   = (v3_pred[..., :NUM_CLASSES] * PCU_VEC[..., :NUM_CLASSES]).sum(axis=-1)

# --- FIGURE 1: 2x2 Spatial Approach Forecasting Grid ---
fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
axes = axes.flatten()
steps = np.arange(len(true_in_pcu))

for n_idx in range(NUM_NODES):
    ax = axes[n_idx]
    ax.plot(steps, true_in_pcu[:, 0, n_idx] * 12.0, label="Observed Flow (RF-DETR)", color="#0969DA", lw=1.9)
    ax.plot(steps, v3_in_pcu[:, 0, n_idx] * 12.0,   label="Proposed Corr-TGCN (v3)", color="#1A7F37", lw=1.9, ls="--")
    ax.set_title(f"Approach: {NODE_NAMES[n_idx]}", fontweight="bold")
    ax.set_ylabel("Hourly Flow Rate (PCU/hr)")
    if n_idx >= 2:
        ax.set_xlabel("Chronological Test Interval (5-min Steps)")
    ax.legend(frameon=True, facecolor="white", fontsize=9.5)

plt.suptitle("Authentic Test-Set PCU Flow Forecasting — Proposed Correlation-Weighted T-GCN (v3)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "Fig1_v3_Spatial_PCU_Grid.png"), dpi=300, bbox_inches="tight")
plt.show()

# --- FIGURE 2: Multi-Horizon Benchmark Bar Chart ---
models_plot = list(summary_mean_df.index)
x = np.arange(len(models_plot))
w = 0.25

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.bar(x - w, summary_mean_df["H1_5min_PCU_MAE"],  w, label="5-min Horizon",  color="#2DA44E")
ax.bar(x,     summary_mean_df["H2_10min_PCU_MAE"], w, label="10-min Horizon", color="#0969DA")
ax.bar(x + w, summary_mean_df["H3_15min_PCU_MAE"], w, label="15-min Horizon", color="#BF3989")
ax.set_xticks(x)
ax.set_xticklabels(models_plot, rotation=20, ha="right")
ax.set_ylabel("PCU Mean Absolute Error (PCU / 5-min)")
ax.set_title("3-Seed Multi-Horizon Forecasting Error Comparison Across 7 Models", fontweight="bold")
ax.legend(frameon=True, facecolor="white")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "Fig2_v3_Horizon_Benchmarks.png"), dpi=300, bbox_inches="tight")
plt.show()

# --- FIGURE 3: HCM 2010 Control Delay Comparison ---
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2))
ax = axes[0]
ax.plot(delay_df["Window"], delay_df["Fixed_50_50"],  label=f"Fixed 50/50 (Mean: {delay_df['Fixed_50_50'].mean():.1f}s)",  color="#CF222E", lw=1.7, ls=":")
ax.plot(delay_df["Window"], delay_df["Fixed_65_35"],  label=f"Fixed 65/35 (Mean: {delay_df['Fixed_65_35'].mean():.1f}s)",  color="#D97706", lw=1.8, ls="--")
ax.plot(delay_df["Window"], delay_df["Corr_TGCN_v3"], label=f"Corr-TGCN v3 (Mean: {delay_df['Corr_TGCN_v3'].mean():.1f}s)", color="#1A7F37", lw=2.2)
ax.plot(delay_df["Window"], delay_df["Oracle_Bound"], label=f"Oracle Bound (Mean: {delay_df['Oracle_Bound'].mean():.1f}s)", color="#0969DA", lw=1.4, ls="-.", alpha=0.85)
ax.set_title("HCM 2010 Control Delay Across Unseen Test Windows", fontweight="bold")
ax.set_xlabel("Chronological Test Window Index (15-min Intervals)")
ax.set_ylabel("Average Control Delay (sec/veh)")
ax.legend(frameon=True, facecolor="white", fontsize=9.5)

ax2 = axes[1]
labels_bar = ["Fixed\n50/50", "Fixed\n65/35", "Fixed\n75/25", "Corr-TGCN\n(v3)", "Oracle\nBound"]
vals_bar   = strat_df["Mean_Delay(s/veh)"].values
los_bar    = strat_df["HCM_LOS"].values
colors_bar = ["#CF222E", "#D97706", "#9A6700", "#1A7F37", "#0969DA"]
bars = ax2.bar(np.arange(len(labels_bar)), vals_bar, color=colors_bar, width=0.55, edgecolor="#333333")
ax2.set_xticks(np.arange(len(labels_bar)))
ax2.set_xticklabels(labels_bar)
ax2.set_ylabel("Mean Control Delay (sec/veh)")
ax2.set_title("Signal Strategy Comparison & HCM 2010 LOS", fontweight="bold")
ax2.set_ylim(0, max(vals_bar) * 1.22)
for bar, val, los in zip(bars, vals_bar, los_bar):
    ax2.text(bar.get_x() + bar.get_width()/2, val + max(vals_bar)*0.02,
             f"{val:.1f}s\n(LOS {los})", ha="center", va="bottom", fontweight="bold", fontsize=9.5)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "Fig3_v3_HCM2010_Delay_Comparison.png"), dpi=300, bbox_inches="tight")
plt.show()

print("All v3 tables & publication figures saved to:", OUT_DIR)